# Skenario 2 — Data Balancing & Augmentation
**Tujuan:** Membandingkan 4 strategi augmentasi data  
- **Branch A:** Raw (tanpa augmentasi)
- **Branch B:** Random Oversampling
- **Branch C:** Back-translation (ID→EN→ID)
- **Branch D:** Synonym Replacement

**Task:** Klasifikasi Emosi (5 kelas) dan Sentimen (3 kelas)

---
**Strategi hemat RAM:** Setiap kombinasi (task × branch × model) dilatih satu per satu, hasilnya langsung di-save ke `results_s2.json`.

**🏆 Best model:** Model + tokenizer terbaik per task (across semua branch) di-save ke `/kaggle/working/best_model_s2_{task}/`

## 0. Install & Import

In [1]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("transformers[torch]", "datasets", "accelerate", "scikit-learn",
    "imbalanced-learn", "deep-translator", "nltk", "seaborn", "matplotlib",
    "pandas", "numpy", "torch")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [ ]:
import os, re, json, warnings, random, time, gc
from copy import deepcopy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import nltk

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from deep_translator import GoogleTranslator
from nltk.corpus import wordnet

from tqdm import tqdm

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 1. Load & Preprocessing Dataset

In [3]:
# 1. Load, Merge, & Preprocessing Dataset
PATH_SEKUNDER = "/kaggle/input/datasets/miaikoo/secondary/PRDECT-ID Dataset_clean.csv"
PATH_PRIMER = "/kaggle/input/datasets/miaikoo/primary/data_primer_final_annotated.csv"

# Load kedua dataset
df_sekunder = pd.read_csv(PATH_SEKUNDER)
df_primer = pd.read_csv(PATH_PRIMER)

df_sekunder_cek = pd.read_csv(PATH_SEKUNDER, nrows=5)
df_primer_cek = pd.read_csv(PATH_PRIMER, nrows=5)

print("=== KOLOM DATA SEKUNDER ===")
print(df_sekunder_cek.columns.tolist())
print("\nPreview Data Sekunder:")
display(df_sekunder_cek.head(2))

print("\n" + "="*50 + "\n")

print("=== KOLOM DATA PRIMER ===")
print(df_primer_cek.columns.tolist())
print("\nPreview Data Primer:")
display(df_primer_cek.head(2))

=== KOLOM DATA SEKUNDER ===
['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Customer Review', 'Sentiment', 'Emotion', 'Location_grouped', 'review_text', 'review_char_len', 'review_word_len', 'review_tokens']

Preview Data Sekunder:


,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review,Sentiment,Emotion,Location_grouped,review_text,review_char_len,review_word_len,review_tokens
0,Computers and Laptops,Wireless Keyboard i8 Mini TouchPad Mouse 2.4G ...,Jakarta Utara,53500,4.9,5449,2369,5,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy,Jakarta Utara,Alhamdulillah berfungsi dengan baik. Packaging...,100,14,"['alhamdulillah', 'berfungsi', 'baik', 'packag..."
1,Computers and Laptops,PAKET LISENSI WINDOWS 10 PRO DAN OFFICE 2019 O...,Kota Tangerang Selatan,72000,4.9,2359,1044,5,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy,Kota Tangerang Selatan,"barang bagus dan respon cepat, harga bersaing ...",61,10,"['barang', 'bagus', 'respon', 'cepat', 'harga'..."




=== KOLOM DATA PRIMER ===
['Platform', 'Rating', 'Review', 'Review_Clean', 'shadow_sentiment', 'confidence_sentiment', 'shadow_emotion', 'confidence_emotion']

Preview Data Primer:


,Platform,Rating,Review,Review_Clean,shadow_sentiment,confidence_sentiment,shadow_emotion,confidence_emotion
0,Tokopedia,5,Toko langganan Kantor. Harga tdk terlalu mahal...,Toko langganan Kantor. Harga tdk terlalu mahal...,Positive,0.999448,Happy,0.812548
1,Tokopedia,5,"Seller fast respon, kualitasnya bagus juga unt...","Seller fast respon, kualitasnya bagus juga unt...",Positive,0.999867,Happy,0.962622


In [4]:
# 1. SAMAKAN NAMA KOLOM
df_primer = df_primer.rename(columns={
    "Review_Clean": "Customer Review",
    "shadow_sentiment": "Sentiment",
    "shadow_emotion": "Emotion"
})

# 2. FILTER KOLOM YANG PENTING SAJA
kolom_penting = ["Customer Review", "Sentiment", "Emotion"]
df_sekunder_filtered = df_sekunder[kolom_penting]
df_primer_filtered = df_primer[kolom_penting]

# 3. GABUNGKAN KEDUA DATASET
df = pd.concat([df_primer_filtered, df_sekunder_filtered], ignore_index=True)

# 4. Bersihkan spasi di nama kolom (jika ada)
df.columns = [c.strip() for c in df.columns]

# 5. Fungsi pembersihan teks
def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text) 
    text = re.sub(r"[^\w\s]", " ", text)          
    text = re.sub(r"\d+", " ", text)                
    text = re.sub(r"\s+", " ", text).strip()          
    return text

df["text_clean"] = df["Customer Review"].apply(clean_text)

# 6. Pastikan tidak ada data kosong (NaN) di kolom target setelah digabung
df = df.dropna(subset=["Sentiment", "Emotion"])

# 7. Label Encoding
le_sent = LabelEncoder()
le_emot = LabelEncoder()
df["label_sentiment"] = le_sent.fit_transform(df["Sentiment"])
df["label_emotion"]   = le_emot.fit_transform(df["Emotion"])

print("Shape setelah digabung & difilter:", df.shape)
print("Label Sentimen :", dict(zip(le_sent.classes_, le_sent.transform(le_sent.classes_))))
print("Label Emosi    :", dict(zip(le_emot.classes_, le_emot.transform(le_emot.classes_))))

Shape setelah digabung & difilter: (38121, 6)
Label Sentimen : {'Negative': np.int64(0), 'Positive': np.int64(1)}
Label Emosi    : {'Anger': np.int64(0), 'Fear': np.int64(1), 'Happy': np.int64(2), 'Love': np.int64(3), 'Sadness': np.int64(4)}


## 2. Helper Functions

In [7]:
# salah satu aja
TASKS = {
    "Emotion":   ("text_clean", "label_emotion",   le_emot.classes_),
    #"Sentiment": ("text_clean", "label_sentiment", le_sent.classes_),
}

def make_splits(texts, labels, test_size=0.2, val_size=0.1):
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=test_size, random_state=SEED, stratify=labels)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=val_size/(1-test_size), random_state=SEED, stratify=y_train)
    return X_train, X_val, X_test, y_train, y_val, y_test


def build_hf_dataset(texts, labels, tokenizer, max_len=128):
    enc = tokenizer(list(texts), truncation=True, padding="max_length",
                    max_length=max_len, return_tensors="pt")
    return HFDataset.from_dict({
        "input_ids":      enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels":         list(labels),
    })


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


def train_and_evaluate(model_name, train_ds, val_ds, test_texts, test_labels,
                       tokenizer, num_labels, label_names,
                       output_dir="./tmp_model",
                       num_epochs=3, batch_size=8, lr=2e-5, 
                       weight_decay=0.01, dropout=None,
                       warmup_ratio=0.1, max_len=128,
                       grad_accum=2): 
    import time
    current_batch_size = batch_size
    while True:
        try:
            # Retry loading model dengan network resilience
            for attempt in range(3):
                try:
                    model = AutoModelForSequenceClassification.from_pretrained(
                        model_name, num_labels=num_labels, ignore_mismatched_sizes=True)
                    break
                except Exception as e:
                    if attempt == 2:
                        raise e
                    print(f"  ⚠️ Gagal download model {model_name}, mencoba lagi ({attempt+2}/3) dalam 5 detik... (Error: {e})")
                    time.sleep(5)

            if dropout is not None and hasattr(model.config, "hidden_dropout_prob"):
                model.config.hidden_dropout_prob = dropout
                model.config.attention_probs_dropout_prob = dropout

            args = TrainingArguments(
                output_dir=output_dir,
                num_train_epochs=num_epochs,
                per_device_train_batch_size=current_batch_size,
                per_device_eval_batch_size=current_batch_size,
                gradient_accumulation_steps=grad_accum,  
                learning_rate=lr,
                weight_decay=weight_decay,
                warmup_ratio=warmup_ratio,
                eval_strategy="epoch",
                save_strategy="epoch",
                save_total_limit=1,
                save_only_model=True,
                load_best_model_at_end=True,
                metric_for_best_model="f1_macro",
                logging_steps=50,
                fp16=(DEVICE == "cuda"),
                report_to="none",
                seed=SEED,
            )

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            )

            t0 = time.time()
            trainer.train()
            elapsed = time.time() - t0
            break
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  ⚠️ [OOM] CUDA out of memory dengan batch_size={current_batch_size}.")
                if 'trainer' in locals():
                    del trainer
                if 'model' in locals():
                    del model
                free_memory()
                current_batch_size = current_batch_size // 2
                
                grad_accum = grad_accum * 2 

                if current_batch_size < 2: 
                    print("  ❌ [OOM] Batch size sudah minimum (< 2). Gagal melatih model ini.")
                    raise e
                print(f"  🔄 Mencoba ulang dengan batch_size={current_batch_size} dan grad_accum={grad_accum}...")
            else:
                raise e

    test_ds = build_hf_dataset(test_texts, test_labels, tokenizer, max_len)
    preds_raw = trainer.predict(test_ds)
    preds_logits = preds_raw.predictions
    if isinstance(preds_logits, tuple):
        preds_logits = preds_logits[0]
    preds = np.argmax(preds_logits, axis=-1)

    metrics = {
        "accuracy":    round(accuracy_score(test_labels, preds), 4),
        "f1_macro":    round(f1_score(test_labels, preds, average="macro"), 4),
        "f1_weighted": round(f1_score(test_labels, preds, average="weighted"), 4),
        "train_time_s": round(elapsed, 1),
    }
    report = classification_report(test_labels, preds, target_names=label_names)
    return trainer, metrics, report, preds


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def save_best_model(trainer, tokenizer, task_name, model_alias, model_name,
                    branch_name, metrics, label_names, skenario="2"):
    """Simpan model + tokenizer + metadata ke folder best_model_s{skenario}_{task}."""
    best_dir = f"/kaggle/working/best_model_s{skenario}_{task_name}"
    os.makedirs(best_dir, exist_ok=True)
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)
    meta = {
        "skenario": skenario,
        "task": task_name,
        "model_alias": model_alias,
        "model_name": model_name,
        "branch": branch_name,
        "label_names": list(label_names),
        "metrics": metrics,
    }
    with open(f"{best_dir}/best_model_info.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  🏆 [BEST MODEL] {model_alias} @ {branch_name} (f1_macro={metrics['f1_macro']:.4f}) → {best_dir}")

## 3. Augmentation Functions

In [10]:
# ── Branch B: Random Oversampling ─────────────────────────────────────
def oversample_data(texts, labels):
    """Duplikasi sampel kelas minoritas."""
    ros = RandomOverSampler(random_state=SEED)
    idx = np.arange(len(texts)).reshape(-1, 1)
    idx_res, labels_res = ros.fit_resample(idx, labels)
    texts_res = texts[idx_res.flatten()]
    return texts_res, labels_res


# ── Branch C: Back-translation ID → EN → ID ───────────────────────────
def back_translate(text: str) -> str:
    try:
        en = GoogleTranslator(source="id", target="en").translate(text[:500])
        id_ = GoogleTranslator(source="en", target="id").translate(en)
        time.sleep(0.5) 
        return id_ if id_ else text
    except Exception as e:
        print(f"    [Translate Error]: {e}") 
        return text

def augment_backtranslation(texts, labels, min_count=None, save_path="/kaggle/working/branch_c_backup.csv"):
    if os.path.exists(save_path):
        print(f"  ✅ [CACHE] File backup ditemukan! Langsung load dari {save_path}...")
        df_backup = pd.read_csv(save_path)
        return df_backup["text"].values, df_backup["label"].values

    if min_count is None:
        cnt = Counter(labels)
        min_count = max(cnt.values())

    aug_texts, aug_labels = list(texts), list(labels)
    cnt = Counter(labels)

    for cls, count in cnt.items():
        need = min_count - count
        if need <= 0:
            continue
        
        cls_idx = np.where(labels == cls)[0]
        sample_idx = np.random.choice(cls_idx, size=need, replace=(need > len(cls_idx)))
        
        print(f"  Back-translating {need} samples for class {cls}...")
        for i in tqdm(sample_idx, desc=f"Translate {cls}"):
            aug_texts.append(back_translate(texts[i]))
            aug_labels.append(cls)
            
        # AUTO-SAVE
        df_save = pd.DataFrame({"text": aug_texts, "label": aug_labels})
        df_save.to_csv(save_path, index=False)
        print(f"  💾 [SAVED] Progress diamankan ke {save_path}")

    return np.array(aug_texts), np.array(aug_labels)


# ── Branch D: Synonym Replacement ─────────────────────────────────────
def get_synonym(word: str) -> str:
    syns = wordnet.synsets(word, lang='ind')
    for syn in syns:
        for lemma in syn.lemmas(lang='ind'):
            if lemma.name() != word:
                return lemma.name().replace("_", " ")
    return word

def synonym_replace(text: str, n: int = 2) -> str:
    words = text.split()
    if len(words) < 3:
        return text
    indices = random.sample(range(len(words)), min(n, len(words)))
    for i in indices:
        words[i] = get_synonym(words[i])
    return " ".join(words)

def augment_synonym(texts, labels, min_count=None):
    if min_count is None:
        cnt = Counter(labels)
        min_count = max(cnt.values())

    aug_texts, aug_labels = list(texts), list(labels)
    cnt = Counter(labels)

    for cls, count in cnt.items():
        need = min_count - count
        if need <= 0:
            continue
        cls_idx = np.where(labels == cls)[0]
        sample_idx = np.random.choice(cls_idx, size=need, replace=(need > len(cls_idx)))
        for i in sample_idx:
            aug_texts.append(synonym_replace(texts[i]))
            aug_labels.append(cls)

    return np.array(aug_texts), np.array(aug_labels)

---
# SKENARIO 2 — Training Loop

Setiap model × branch × task dilatih satu per satu → hasil di-save ke JSON → **best model per task disimpan** → memory dibebaskan.

In [11]:
FAST_MODE_S2   = False   # True untuk quick test
SKIP_BACKTRANS = False    # True untuk skip back-translation (lambat)
NUM_EPOCHS_S2  = 3
BATCH_SIZE_S2  = 8
LR_S2 = 2e-5

S2_MODELS = {
    #"DistilBERT": "distilbert-base-multilingual-cased",
    "IndoBERT":  "indobenchmark/indobert-base-p1"
}

RESULTS_FILE = "/kaggle/working/results_s2.json"

# Load hasil yang sudah ada (untuk resume)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        S2_RESULTS = json.load(f)
    print(f"Resume dari: {list(S2_RESULTS.keys())}")
else:
    S2_RESULTS = {}
    print("Mulai dari awal.")

# Tracker best model per task
BEST_MODEL_TRACKER = {}

Mulai dari awal.


In [12]:
# Siapkan semua augmented data terlebih dahulu (tidak perlu GPU)
S2_BRANCHES = {}
S2_SPLITS   = {}

for task_name, (text_col, label_col, label_names) in TASKS.items():
    print(f"\n=================== PREPARING DATA FOR TASK: {task_name} ===================")
    
    texts_all  = df[text_col].values
    labels_all = df[label_col].values
    
    if FAST_MODE_S2:
        idx = np.random.choice(len(texts_all), size=int(len(texts_all)*0.3), replace=False)
        texts_all, labels_all = texts_all[idx], labels_all[idx]

    X_train_raw, X_val, X_test, y_train_raw, y_val, y_test = make_splits(texts_all, labels_all)
    S2_SPLITS[task_name] = (X_train_raw, X_val, X_test, y_train_raw, y_val, y_test)

    print(f"[{task_name}] Base split → Train: {len(X_train_raw)}, Val: {len(X_val)}, Test: {len(X_test)}")

    branches = {
        "A_Raw": (X_train_raw.copy(), y_train_raw.copy()),
    }

    print(f"[{task_name}] [Branch B] Oversampling...")
    X_B, y_B = oversample_data(X_train_raw, y_train_raw)
    branches["B_Oversample"] = (X_B, y_B)
    print(f"  → Setelah oversample: {len(X_B)}")

    if not SKIP_BACKTRANS:
        print(f"[{task_name}] [Branch C] Back-translation...")
        X_C, y_C = augment_backtranslation(X_train_raw, y_train_raw)
        branches["C_BackTrans"] = (X_C, y_C)
        print(f"  → Setelah back-trans: {len(X_C)}")
    else:
        print(f"[{task_name}] [Branch C] Skipped (SKIP_BACKTRANS=True)")

    print(f"[{task_name}] [Branch D] Synonym replacement...")
    X_D, y_D = augment_synonym(X_train_raw, y_train_raw)
    branches["D_Synonym"] = (X_D, y_D)
    print(f"  → Setelah synonym aug: {len(X_D)}")
    
    S2_BRANCHES[task_name] = branches

print("\nSemua data augmentation selesai!")


=================== PREPARING DATA FOR TASK: Emotion ===================
[Emotion] Base split → Train: 26684, Val: 3812, Test: 7625
[Emotion] [Branch B] Oversampling...
  → Setelah oversample: 44585
[Emotion] [Branch C] Back-translation...
  Back-translating 5927 samples for class 0...


KeyboardInterrupt: 

  Back-translating 750 samples for class 0...


  Back-translating 595 samples for class 1...


  → Setelah back-trans: 6195
[Emotion] [Branch D] Synonym replacement...


  → Setelah synonym aug: 6195

=================== PREPARING DATA FOR TASK: Sentiment ===================
[Sentiment] Base split → Train: 3780, Val: 540, Test: 1080
[Sentiment] [Branch B] Oversampling...
  → Setelah oversample: 3950
[Sentiment] [Branch C] Back-translation...
  Back-translating 170 samples for class 1...


  → Setelah back-trans: 3950
[Sentiment] [Branch D] Synonym replacement...
  → Setelah synonym aug: 3950

Semua data augmentation selesai!


In [8]:
# Training loop — model dilatih satu per satu
for task_name, (text_col, label_col, label_names) in TASKS.items():
    print(f"\n=================== TRAINING SKENARIO 2 FOR TASK: {task_name} ===================")
    
    num_labels = len(label_names)
    X_train_raw, X_val, X_test, y_train_raw, y_val, y_test = S2_SPLITS[task_name]
    branches = S2_BRANCHES[task_name]

    for branch_name, (X_train, y_train) in branches.items():
        for model_alias, model_name in S2_MODELS.items():
            result_key = f"{task_name}|{branch_name}|{model_alias}"
            
            # Skip jika sudah ada — tetap update tracker
            if task_name in S2_RESULTS:
                if branch_name in S2_RESULTS[task_name]:
                    if model_alias in S2_RESULTS[task_name][branch_name]:
                        print(f"  [SKIP] {result_key} sudah ada.")
                        existing_f1 = S2_RESULTS[task_name][branch_name][model_alias]["metrics"]["f1_macro"]
                        if task_name not in BEST_MODEL_TRACKER or existing_f1 > BEST_MODEL_TRACKER[task_name]["f1_macro"]:
                            BEST_MODEL_TRACKER[task_name] = {"f1_macro": existing_f1, "model_alias": model_alias,
                                                             "branch": branch_name, "model_name": model_name}
                        continue

            print(f"\n>>> Task: {task_name} | {branch_name} — {model_alias}")

            try:
                # Retry loading tokenizer dengan network resilience
                import time
                for attempt in range(3):
                    try:
                        tokenizer = AutoTokenizer.from_pretrained(model_name)
                        break
                    except Exception as e:
                        if attempt == 2:
                            raise e
                        print(f"  ⚠️ Gagal download tokenizer {model_name}, mencoba lagi ({attempt+2}/3) dalam 5 detik... (Error: {e})")
                        time.sleep(5)

                train_ds = build_hf_dataset(X_train, y_train, tokenizer)
                val_ds   = build_hf_dataset(X_val,   y_val,   tokenizer)

                output_dir = f"/kaggle/working/s2_{task_name}_{branch_name}_{model_alias}"

                trainer, metrics, report, preds = train_and_evaluate(
                    model_name, train_ds, val_ds, X_test, y_test,
                    tokenizer, num_labels, label_names,
                    output_dir=output_dir,
                    num_epochs=NUM_EPOCHS_S2,
                    batch_size=BATCH_SIZE_S2,
                    lr=LR_S2,
                )

                print(f"  F1 Macro: {metrics['f1_macro']} | Accuracy: {metrics['accuracy']}")

                # Simpan hasil
                if task_name not in S2_RESULTS:
                    S2_RESULTS[task_name] = {}
                if branch_name not in S2_RESULTS[task_name]:
                    S2_RESULTS[task_name][branch_name] = {}

                S2_RESULTS[task_name][branch_name][model_alias] = {
                    "metrics": metrics,
                    "report":  report,
                }

                # ── Cek & simpan best model ──────────────────────────────────
                current_f1 = metrics["f1_macro"]
                if task_name not in BEST_MODEL_TRACKER or current_f1 > BEST_MODEL_TRACKER[task_name]["f1_macro"]:
                    BEST_MODEL_TRACKER[task_name] = {
                        "f1_macro": current_f1,
                        "model_alias": model_alias,
                        "branch": branch_name,
                        "model_name": model_name,
                    }
                    save_best_model(trainer, tokenizer, task_name, model_alias, model_name,
                                    branch_name, metrics, label_names, skenario="2")
                else:
                    print(f"  (bukan best: {current_f1:.4f} < {BEST_MODEL_TRACKER[task_name]['f1_macro']:.4f})")

                with open(RESULTS_FILE, "w") as f:
                    json.dump(S2_RESULTS, f, indent=2)
                print(f"  [SAVED] → {RESULTS_FILE}")

                # Bebaskan memori
                del trainer, tokenizer, train_ds, val_ds
                free_memory()

                # Hapus folder output tiap iterasi biar ngga OOM
                import shutil
                if os.path.exists(output_dir):
                    shutil.rmtree(output_dir, ignore_errors=True)

            except Exception as e:
                print(f"  [ERROR] {result_key}: {e}")
                free_memory()
                import shutil
                if 'output_dir' in locals() and os.path.exists(output_dir):
                    shutil.rmtree(output_dir, ignore_errors=True)

print("\n✅ Skenario 2 selesai!")
print("\n🏆 Best Models per Task:")
for task, info in BEST_MODEL_TRACKER.items():
    print(f"  {task}: {info['model_alias']} @ {info['branch']} (f1_macro={info['f1_macro']:.4f}) → /kaggle/working/best_model_s2_{task}/")


=================== TRAINING SKENARIO 2 FOR TASK: Emotion ===================

>>> Task: Emotion | A_Raw — BERT


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.309661,0.274494,0.953704,0.953530,0.953656
2,0.168274,0.213652,0.968519,0.968451,0.968516


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [ERROR] Sentiment|D_Synonym|cahya-DistilBERT: Error while serializing: I/O error: No space left on device (os error 28)



✅ Skenario 2 selesai!

🏆 Best Models per Task:
  Emotion: IndoBERT @ A_Raw (f1_macro=0.6617) → /kaggle/working/best_model_s2_Emotion/
  Sentiment: IndoBERT @ C_BackTrans (f1_macro=0.9842) → /kaggle/working/best_model_s2_Sentiment/
